# Parte 2

O objetivo desta parte do trabalho é analisar o **comportamento dos índices** das tabelas do SGBD através do exame e análise das tabelas de estatísticas para consultas SQL sobre uma tabela criada com dados aleatórios.

Configuração inicial:

In [1]:
# Conectar ao banco de dados PostgreSQL
import psycopg2

config = {
    'dbname': 'icomp',
    'user': 'icomp',
    'password': 'icomp123',
}

conn = psycopg2.connect(**config)
cur = conn.cursor()

# Configurar rich
from rich.table import Table
from rich.console import Console

console = Console()

---
## Tarefa 5
**Preparação da Tabela Aleatória**

Criar uma tabela com uma chave simples e alguns dados de exemplo. Cada valor de chave é um número incremental e está associado a com valores que variam de 0 até 10:

In [2]:
cur.execute("""
DROP TABLE IF EXISTS t;
CREATE TABLE t (
    k SERIAL PRIMARY KEY,
    v INTEGER
);  
""")

cur.execute("""
INSERT INTO t(v)
SELECT trunc(random() * 11)      -- valores entre 0 e 10
FROM generate_series(1, 100000); -- 100 mil
""")

### O que entregar

Imprimir os valores das 10 primeiras tuplas da tabela, ordenando por k:

In [3]:
cur.execute("""
SELECT * FROM t ORDER BY k LIMIT 10;
""")
rows = cur.fetchall()

print('\n'.join(f'Tupla(k: {row[0]:>2}, v: {row[1]})' for row in rows))

Tupla(k:  1, v: 10)
Tupla(k:  2, v: 5)
Tupla(k:  3, v: 0)
Tupla(k:  4, v: 5)
Tupla(k:  5, v: 10)
Tupla(k:  6, v: 4)
Tupla(k:  7, v: 9)
Tupla(k:  8, v: 8)
Tupla(k:  9, v: 2)
Tupla(k: 10, v: 3)


---
## Tarefa 6
**Páginas criadas**

Verifique quantas páginas com blocos foram criadas para a tabela da Tarefa 5.

Comando: `SELECT relname, relpages, reltuples FROM pg_class WHERE relname='t';`

### O que entregar

Imprimir o resutlado do comando SQL

Antes de executar a consulta, é necessário atualizar as estatísticas da tabela com o comando `ANALYZE t;` para garantir que os valores retornados estejam corretos

In [4]:
cur.execute("ANALYZE t;")
conn.commit()

In [5]:
# Executa a consulta
cur.execute("""
    SELECT relname, relpages, reltuples
    FROM pg_class
    WHERE relname = 't';
""")

name, npages, ntuples = cur.fetchall()[0]

table = Table(title="Estatísticas da Tabela t")
table.add_column("Nome", justify="left")
table.add_column("Páginas", justify="right")
table.add_column("Tuplas", justify="right")

table.add_row(str(name), str(npages), str(ntuples))

# Imprime
console.print(table)

  Estatísticas da Tabela t   
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓
┃ Nome ┃ Páginas ┃   Tuplas ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩
│ t    │     443 │ 100000.0 │
└──────┴─────────┴──────────┘

---
## Tarefa 7
**Blocos**

Verifique quantos blocos foram efetivamente usados numa consulta

Comando:
```sql
SELECT pg_sleep(1);
\pset x on
SELECT * FROM pg_stats WHERE tablename='t';
SELECT pg_stat_reset();
\pset x off
```

- `SELECT pg_sleep(1);` faz o servidor dormir por 1 segundo; usado para sincronizar, dar tempo para o autovacuum ou apenas demonstrar espera
- `\pset x on` ativa o modo expandido do psql, exibindo cada coluna em formato vertical, facilitando a leitura quando a linha tem muitas colunas
- `SELECT * FROM pg_stats WHERE tablename='t';` mostra estatísticas coletadas pelo ANALYZE sobre colunas de tabelas, filtrando para mostrar somente informações referentes à tabela 't'
- `SELECT pg_stat_reset();` reseta estatísticas de todo o cluster, incluindo pg_stat_all_tables, desde que o usuário seja superuser. Não afeta pg_stats, pois pg_stats depende das estatísticas coletadas pelo ANALYZE
- `\pset x off` volta o psql ao modo de exibição normal (tabelado horizontal)


### O que entregar

Imprimir o resultado do comando SQL

⚠️ Os comandos foram feitos via terminal

In [6]:
!psql -d icomp -U icomp -c "SELECT pg_sleep(1);"

 pg_sleep 
----------
 
(1 row)



In [7]:
!psql -d icomp -U icomp -c "\pset x on" -c "SELECT * FROM pg_stats WHERE tablename='t';" -c "SELECT pg_stat_reset();" -c "\pset x off"

Expanded display is on.
-[ RECORD 1 ]----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
schemaname             | public
tablename              | t
attname                | k
inherited              | f
null_frac              | 0
avg_width              | 4
n_distinct             | -1
most_common_vals       | 
most_common_freqs      | 
histogram_bounds       | {2,974,1977,2987,3982,4988,6028,6989,7994,9100,10171,11207,12214,13177,14171,15197,16

---
## Tarefa 8
**Índice**

1. Crie um índice para o atributo 'v' e realize consultas e criação de índice

    a. Qual o tempo gasto para realizar uma consulta para um valor (lendo a tabela 100.000 tuplas)? 
    
    b. Qual o tempo gasto para recriar um índice para o atributo 'v'?

2. Remova a tabela 't' e crie novamente com 1.000.000 de tuplas

    a. Qual o tempo gasto para realizar uma consulta para um valor específico? 
    
    b. Qual o tempo gasto para recriar um índice para o atributo 'v'?

### O que entregar

**Relatório com o resultado das perguntas**

Antes de tudo, criar uma função para medir o tempo gasto em consultas

In [8]:
import time

def time_query(query, params=None) -> tuple:
    start = time.time()
    cur.execute(query, params)
    result = cur.fetchall()
    end = time.time()
    
    total_time = end - start
    return total_time, result

In [9]:
conn.rollback()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()

#### 1. Crie um índice para o atributo 'v' e realize consultas e criação de índice


##### **A)** Qual o tempo gasto para realizar uma consulta para um valor (lendo a tabela 100.000 tuplas)?

Sem índice

In [10]:
query = "SELECT * FROM t WHERE v = %s;"
time_no_index, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo sem índice:[/] "
              f"[white]{time_no_index:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

Tempo sem índice: 0.019396 segundos

Linhas retornadas: 8951

In [11]:
# Criar índice
cur.execute("CREATE INDEX IF NOT EXISTS idx_t_v ON t(v);")
conn.commit()

Com índice

In [12]:
time_with_index, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo com índice:[/] "
              f"[white]{time_with_index:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

Tempo com índice: 0.003343 segundos

Linhas retornadas: 8951

##### **B)** Qual o tempo gasto para recriar um índice para o atributo 'v'?

In [13]:
conn.rollback()

# Remover índice
t0 = time.perf_counter()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()
t1 = time.perf_counter()
drop_time = t1 - t0

# Criar novamente
t0 = time.perf_counter()
cur.execute("CREATE INDEX idx_t_v ON t(v);")
conn.commit()
t1 = time.perf_counter()
recreate_time = t1 - t0

console.print("[#A8EFFF]Tempo para DROP INDEX:[/] "
              f"[white]{drop_time:.6f} segundos[/white]")
console.print("[#A8EFFF]Tempo para recriar índice:[/] "
              f"[white]{recreate_time:.6f} segundos[/white]")

Tempo para DROP INDEX: 0.001234 segundos

Tempo para recriar índice: 0.033179 segundos

#### 2. Remova a tabela 't' e crie novamente com 1.000.000 de tuplas

In [14]:
cur.execute("""
DROP TABLE IF EXISTS t;
CREATE TABLE t (
    k SERIAL PRIMARY KEY,
    v INTEGER
);
""")

cur.execute("""
INSERT INTO t(v)
SELECT trunc(random() * 11)
FROM generate_series(1, 1000000);
""")

In [15]:
conn.rollback()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()

##### **A)** Qual o tempo gasto para realizar uma consulta para um valor específico?

Sem Índice

In [16]:
query = "SELECT * FROM t WHERE v = %s;"
time_no_index_1M, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo sem índice (1.000.000 tuplas):[/] "
              f"[white]{time_no_index_1M:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")


Tempo sem índice (1.000.000 tuplas): 0.008184 segundos

Linhas retornadas: 8951

In [17]:
# Criar índice
cur.execute("CREATE INDEX IF NOT EXISTS idx_t_v ON t(v);")
conn.commit()

Com Índice

In [18]:
time_with_index_1M, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo com índice (1.000.000 tuplas):[/] "
              f"[white]{time_with_index_1M:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

Tempo com índice (1.000.000 tuplas): 0.002687 segundos

Linhas retornadas: 8951

##### **B)** Qual o tempo gasto para recriar um índice para o atributo 'v'?

In [19]:
# Remover índice
t0 = time.perf_counter()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()
t1 = time.perf_counter()
drop_time = t1 - t0

# Criar novamente
t0 = time.perf_counter()
cur.execute("CREATE INDEX idx_t_v ON t(v);")
conn.commit()
t1 = time.perf_counter()
recreate_time = t1 - t0

console.print("[#A8EFFF]Tempo para DROP INDEX (1mi tuplas):[/] "
              f"[white]{drop_time:.6f} segundos[/white]")
console.print("[#A8EFFF]Tempo para recriar índice (1mi tuplas):[/] "
              f"[white]{recreate_time:.6f} segundos[/white]")

Tempo para DROP INDEX (1mi tuplas): 0.001251 segundos

Tempo para recriar índice (1mi tuplas): 0.031637 segundos

## Tarefa 9
**Fill factor**

Quando se cria um novo índice, nem toda entrada no bloco do índice é usada. Um espaço livre é deixado, conforme o parâmetro fillfactor. 

Crie novos índices usando fillfactor=60,80,90 e 100. Analise o desempenho de suas consultas usando as mesmas condições da Tarefa *8*

```sql
ALTER TABLE foo SET ( fillfactor = XX );
VACUUM FULL foo;
```

- `VACUUM FULL foo;` reescreve a tabela inteira para liberar espaço não utilizado e otimizar o armazenamento


### O que entregar
**Relatório com o resultado das perguntas**

Primeiramente, garantir que o índice anterior foi removido

In [20]:
conn.rollback()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
conn.commit()

In [37]:
def vacuum_full():
    c2 = psycopg2.connect(conn.dsn)
    c2.autocommit = True
    cur2 = c2.cursor()
    cur2.execute("VACUUM FULL t;")
    cur2.close()
    c2.close()

def alter_table_fillfactor(fillfactor: int):
    conn.commit()
    cur.execute(f"ALTER TABLE t SET (fillfactor = {fillfactor});")
    conn.commit()
    vacuum_full()  # precisa ser isolado

def create_index_with_fillfactor(fillfactor: int):
    conn.commit()

    cur.execute("DROP INDEX IF EXISTS idx_t_v;")
    conn.commit()

    vacuum_full()

    cur.execute(f"""
        CREATE INDEX idx_t_v ON t(v)
        WITH (fillfactor = {fillfactor});
    """)
    conn.commit()

def test_fillfactor(fillfactor: int):
    alter_table_fillfactor(fillfactor)

    # teste sem índice
    conn.commit()
    cur.execute("DROP INDEX IF EXISTS idx_t_v;")
    conn.commit()

    time_no_index_ff, _ = time_query(
        "SELECT * FROM t WHERE v = %s;", (5,)
    )

    # teste com índice
    create_index_with_fillfactor(fillfactor)

    time_with_index_ff, _ = time_query(
        "SELECT * FROM t WHERE v = %s;", (5,)
    )

    return {
        "fillfactor": fillfactor,
        "time_no_index": time_no_index_ff,
        "time_with_index": time_with_index_ff,
    }

Realizar a comparação:

In [44]:
fill_factors = [60, 80, 90, 100]
results = []

for ff in fill_factors:
    console.print(f"Testando fillfactor = {ff}")
    results.append(test_fillfactor(ff))

table = Table(title="Comparação de desempenho por Fillfactor")
table.add_column("Fillfactor", justify="center")
table.add_column("Tempo sem índice (ms)", justify="right")
table.add_column("Tempo com índice (ms)", justify="right")

for r in results:
    table.add_row(
        str(r["fillfactor"]),
        f"{r['time_no_index'] * 1000:.3f}",
        f"{r['time_with_index'] * 1000:.3f}",
    )

console.print(table)

Testando fillfactor = 60

Testando fillfactor = 80

Testando fillfactor = 90

Testando fillfactor = 100

           Comparação de desempenho por Fillfactor            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Fillfactor ┃ Tempo sem índice (ms) ┃ Tempo com índice (ms) ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│     60     │                 5.771 │                 2.870 │
│     80     │                 5.330 │                 2.713 │
│     90     │                 5.376 │                 4.047 │
│    100     │                 5.419 │                 2.430 │
└────────────┴───────────────────────┴───────────────────────┘

## Tarefa 10
**Utilize índices com ordem DESC**

Repita os testes da Tarefa 8 e 9 usando índices com ordem DESC. Avalie e registre o resultado

Comando:
```sql
CREATE INDEX idx_t_v_desc ON t(v DESC NULLS FIRST);
```

### O que entregar
**Relatório com o resultado da avaliação e uma análise dos resultados.**

In [45]:
conn.rollback()
cur.execute("DROP INDEX IF EXISTS idx_t_v;")
cur.execute("DROP INDEX IF EXISTS idx_t_v_desc;")
conn.commit()

Funções auxiliares

In [46]:
def create_index_desc():
    conn.commit()
    cur.execute("DROP INDEX IF EXISTS idx_t_v_desc;")
    conn.commit()

    vacuum_full()

    cur.execute("""
        CREATE INDEX idx_t_v_desc
        ON t(v DESC NULLS FIRST);
    """)
    conn.commit()


def create_index_desc_with_fillfactor(fillfactor: int):
    conn.commit()
    cur.execute("DROP INDEX IF EXISTS idx_t_v_desc;")
    conn.commit()

    vacuum_full()

    cur.execute(f"""
        CREATE INDEX idx_t_v_desc
        ON t(v DESC NULLS FIRST)
        WITH (fillfactor = {fillfactor});
    """)
    conn.commit()

#### 1. Repetir os testes da Tarefa 8 usando índices com ordem DESC

In [50]:
conn.rollback()
cur.execute("DROP TABLE IF EXISTS t;")
cur.execute("""
CREATE TABLE t (
    k SERIAL PRIMARY KEY,
    v INTEGER
);""")  

cur.execute("""
INSERT INTO t(v)
SELECT trunc(random() * 11)      -- valores entre 0 e 10
FROM generate_series(1, 100000); -- 100 mil
""")
conn.commit()

In [51]:
# sem índice
query = "SELECT * FROM t WHERE v = %s;"
time_no_index_desc, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo sem índice (DESC):[/] "
              f"[white]{time_no_index_desc:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

# criar índice DESC
create_index_desc()
time_with_index_desc, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo com índice DESC:[/] "
              f"[white]{time_with_index_desc:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

# recriar índice (DROP + CREATE)
t0 = time.perf_counter()
cur.execute("DROP INDEX IF EXISTS idx_t_v_desc;")
conn.commit()
t_drop = time.perf_counter() - t0

t0 = time.perf_counter()
create_index_desc()
t_create = time.perf_counter() - t0

console.print("[#A8EFFF]Tempo para DROP INDEX DESC:[/] "
              f"[white]{t_drop:.6f} segundos[/white]")
console.print("[#A8EFFF]Tempo para recriar índice DESC:[/] "
              f"[white]{t_create:.6f} segundos[/white]")

Tempo sem índice (DESC): 0.023057 segundos

Linhas retornadas: 9170

Tempo com índice DESC: 0.002426 segundos

Linhas retornadas: 9170

Tempo para DROP INDEX DESC: 0.001398 segundos

Tempo para recriar índice DESC: 0.119497 segundos

Refazer os testes com 1.000.000 de tuplas

In [52]:
conn.rollback()
cur.execute("DROP TABLE IF EXISTS t;")
cur.execute("""
CREATE TABLE t (
    k SERIAL PRIMARY KEY,
    v INTEGER
);""")

cur.execute("""
INSERT INTO t(v)
SELECT trunc(random() * 11)
FROM generate_series(1, 1000000);
""")
conn.commit()

In [53]:
# sem índice
query = "SELECT * FROM t WHERE v = %s;"
time_no_index_desc, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo sem índice (DESC):[/] "
              f"[white]{time_no_index_desc:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

# criar índice DESC
create_index_desc()
time_with_index_desc, rows = time_query(query, (5,))

console.print("[#A8EFFF]Tempo com índice DESC:[/] "
              f"[white]{time_with_index_desc:.6f} segundos[/white]")
console.print("[#A8EFFF]Linhas retornadas:[/] "
              f"[white]{len(rows)}[/white]")

# recriar índice (DROP + CREATE)
t0 = time.perf_counter()
cur.execute("DROP INDEX IF EXISTS idx_t_v_desc;")
conn.commit()
t_drop = time.perf_counter() - t0

t0 = time.perf_counter()
create_index_desc()
t_create = time.perf_counter() - t0

console.print("[#A8EFFF]Tempo para DROP INDEX DESC:[/] "
              f"[white]{t_drop:.6f} segundos[/white]")
console.print("[#A8EFFF]Tempo para recriar índice DESC:[/] "
              f"[white]{t_create:.6f} segundos[/white]")

Tempo sem índice (DESC): 0.050323 segundos

Linhas retornadas: 90785

Tempo com índice DESC: 0.032962 segundos

Linhas retornadas: 90785

Tempo para DROP INDEX DESC: 0.011797 segundos

Tempo para recriar índice DESC: 1.690086 segundos

#### 2. Repetir os testes da Tarefa 9 usando índices com ordem DESC

In [55]:
def test_fillfactor_desc(fillfactor: int):
    alter_table_fillfactor(fillfactor)

    conn.commit()
    cur.execute("DROP INDEX IF EXISTS idx_t_v_desc;")
    conn.commit()

    time_no_index_ff, _ = time_query("SELECT * FROM t WHERE v = %s;", (5,))

    create_index_desc_with_fillfactor(fillfactor)
    time_with_index_ff, _ = time_query("SELECT * FROM t WHERE v = %s;", (5,))

    return {
        "fillfactor": fillfactor,
        "time_no_index": time_no_index_ff,
        "time_with_index": time_with_index_ff,
    }

In [56]:
fill_factors = [60, 80, 90, 100]
results_desc = []

for ff in fill_factors:
    console.print(f"Testando fillfactor (DESC) = {ff}")
    results_desc.append(test_fillfactor_desc(ff))

table = Table(title="Comparação de desempenho por Fillfactor (índice DESC)")
table.add_column("Fillfactor", justify="center")
table.add_column("Tempo sem índice (ms)", justify="right")
table.add_column("Tempo com índice DESC (ms)", justify="right")

for r in results_desc:
    table.add_row(
        str(r["fillfactor"]),
        f"{r['time_no_index'] * 1000:.3f}",
        f"{r['time_with_index'] * 1000:.3f}",
    )

console.print(table)

Testando fillfactor (DESC) = 60

Testando fillfactor (DESC) = 80

Testando fillfactor (DESC) = 90

Testando fillfactor (DESC) = 100

       Comparação de desempenho por Fillfactor (índice DESC)       
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Fillfactor ┃ Tempo sem índice (ms) ┃ Tempo com índice DESC (ms) ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     60     │                67.723 │                     35.011 │
│     80     │                65.891 │                     33.275 │
│     90     │                65.080 │                     29.764 │
│    100     │                74.071 │                     33.626 │
└────────────┴───────────────────────┴────────────────────────────┘

#### Análise comparativa entre índices ASC e DESC

In [57]:
table = Table(title="Comparação entre Índices ASC e DESC")
table.add_column("Fillfactor", justify="center")
table.add_column("Tempo com índice ASC (ms)", justify="right")
table.add_column("Tempo com índice DESC (ms)", justify="right")

for ff in fill_factors:
    time_with_index_asc = next(
        r["time_with_index"] for r in results if r["fillfactor"] == ff
    )
    time_with_index_desc = next(
        r["time_with_index"] for r in results_desc if r["fillfactor"] == ff
    )
    table.add_row(
        str(ff),
        f"{time_with_index_asc * 1000:.3f}",
        f"{time_with_index_desc * 1000:.3f}",
    )

console.print(table)

                  Comparação entre Índices ASC e DESC                  
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Fillfactor ┃ Tempo com índice ASC (ms) ┃ Tempo com índice DESC (ms) ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     60     │                     2.870 │                     35.011 │
│     80     │                     2.713 │                     33.275 │
│     90     │                     4.047 │                     29.764 │
│    100     │                     2.430 │                     33.626 │
└────────────┴───────────────────────────┴────────────────────────────┘